# Exercice MCP de A à Z — Serveur « Meteo Africa » · Cahier participant
## Formation IA pour les développeurs · Jour 3 · Orsys

**Objectif :** reconstruire, pièce par pièce, le serveur MCP météo montré en démo (`../server.py`),
le tester comme le ferait Claude, puis le brancher sur Claude Desktop.

**Fil rouge :** une seule question tout au long du notebook —
*« Claude, quelle est la météo à Yaoundé en ce moment ? »*
Chaque module ajoute la pièce qui manque pour que Claude puisse y répondre.

**Comment ça marche ? Vous n'écrivez PAS de code : vous remplacez des trous par des mots.**

Chaque cellule à compléter contient **exactement 5 trous**, écrits `___1___`, `___2___`, `___3___`, `___4___`, `___5___`.

Pour chaque cellule :

1. Lisez le **tableau** juste au-dessus : pour chaque trou, il dit **ce qu'il faut écrire** et propose **2 choix**.
   Un seul des deux est le bon.
2. Dans la cellule de code, repérez la ligne qui contient le trou (le commentaire `# TROU 1 : ...` est sur la même ligne).
3. Effacez `___1___` et écrivez le mot choisi **à la place**, sans rien changer d'autre.
   Si le trou est entre guillemets, par exemple `"___1___"`, **gardez les guillemets** : `"serveur"`.
4. Faites pareil pour les trous 2, 3, 4 et 5.
5. Exécutez la cellule (`Maj + Entrée`), puis exécutez la cellule **Vérification** juste en dessous.

Exemple : la ligne `import ___1___` devient `import requests`.

> **Message `NameError: name '___3___' is not defined` ?** Vous avez oublié de remplacer le trou n°3.
>
> **Attention :** exécutez les cellules **dans l'ordre**, de haut en bas : chaque module réutilise les précédents.

### Plan du notebook

| Module | Ce qu'on construit | Trous |
|---|---|---|
| 0 | Préparation (vérification des installations) | — |
| 1 | Comprendre le vocabulaire MCP (quiz) | 5 |
| 2 | Imports, adresses des API, création du serveur | 5 |
| 3 | Le dictionnaire des codes météo | 5 |
| 4 | Fonction interne : ville → coordonnées GPS | 5 |
| 5 | Fonction interne : coordonnées GPS → météo | 5 |
| 6 | Outil MCP n°1 : `meteo_ville` | 5 |
| 7 | Outil MCP n°2 : `comparer_meteo` | 5 |
| 8 | Outil MCP n°3 : `conseil_tenue` | 5 |
| 9 | Outil MCP n°4 : `meteo_resume` | 5 |
| 10 | Assembler le fichier `mon_serveur.py` | 5 |
| 11 | Tester le serveur avec un vrai client MCP (comme Claude) | 5 |
| 12 | Brancher le serveur sur Claude Desktop | 5 |
| | Exercice final : le récap en phrases | 5 |

## Module 0 — Préparation

Rien à remplir ici : exécutez simplement la cellule (`Maj + Entrée`).
Elle vérifie que les deux bibliothèques nécessaires sont installées et prépare l'outil de vérification des quiz.

In [ ]:
# ── Vérification de l'environnement ─────────────────────────────────────────
import sys                                   # infos sur le Python qui exécute ce notebook
import importlib.util                        # permet de tester si un module est installé sans l'importer
import unicodedata                           # sert à ignorer les accents dans les réponses du quiz
import hashlib                               # sert à vérifier les réponses du quiz sans les afficher en clair

print("Python utilisé :", sys.executable)    # ce chemin servira au Module 12 (config Claude Desktop)

for module in ["requests", "mcp"]:           # les 2 dépendances du serveur (voir ../requirements.txt)
    if importlib.util.find_spec(module):     # find_spec renvoie None si le module est absent
        print(f"OK {module} installé")
    else:
        print(f"ERREUR : {module} manquant → lancez :  pip install {module}")

def verifier(numero, reponse, empreinte):
    """Compare une réponse de quiz à la bonne réponse, stockée sous forme d'empreinte
    (hash) : on peut vérifier sans que la solution soit lisible dans le notebook."""
    propre = str(reponse).strip().lower()                       # on ignore majuscules et espaces
    propre = unicodedata.normalize("NFKD", propre).encode("ascii", "ignore").decode()  # ... et les accents
    ok = hashlib.sha256(propre.encode()).hexdigest()[:12] == empreinte
    print(f"{'OK' if ok else 'ERREUR'} Trou {numero} : {reponse}")
    return ok

## Module 1 — Le vocabulaire MCP en 2 minutes

**MCP (Model Context Protocol)** = une prise standard entre une IA et le monde extérieur.
On l'appelle souvent *« l'USB-C de l'IA »* : un seul format de prise, n'importe quel appareil.

```
   Vous                Claude Desktop                  Notre serveur MCP             Internet
 ┌───────┐  question  ┌──────────────┐  appel d'outil  ┌──────────────────┐  HTTP   ┌────────────┐
 │  Vous │ ─────────> │  CLIENT MCP  │ ──────────────> │  meteo_ville()   │ ──────> │ Open-Meteo │
 └───────┘            └──────────────┘   (via stdio)   └──────────────────┘         └────────────┘
```

- Le **serveur** expose des **outils** (*tools*) : de simples fonctions Python.
- Le **client** (Claude Desktop) lit la **docstring** de chaque outil pour décider *quand* l'appeler.
- Les deux se parlent par l'entrée/sortie standard : le transport **stdio**.
- La météo vient de l'**API** gratuite Open-Meteo, sans clé.

### Exercice 1 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | Le programme que NOUS écrivons, et qui expose des outils, s'appelle le... | `navigateur` ou `serveur` |
| `___2___` | Claude Desktop, qui se connecte à notre programme, joue le rôle de... | `client` ou `serveur` |
| `___3___` | Une fonction Python que Claude a le droit d'appeler s'appelle un... (mot anglais) | `function` ou `tool` |
| `___4___` | Le texte entre triple guillemets sous une fonction, lu par Claude pour savoir quand l'utiliser | `docstring` ou `commentaire` |
| `___5___` | Le nom du mode de communication (entrée/sortie standard) entre Claude et le serveur | `http` ou `stdio` |

In [ ]:
# ── Quiz : complétez chaque phrase avec le bon mot (entre les guillemets) ───

# TROU 1 : Le programme que NOUS écrivons, et qui expose des outils, s'appelle le...
reponse_1 = "___1___"

# TROU 2 : Claude Desktop, qui se connecte à notre programme, joue le rôle de...
reponse_2 = "___2___"

# TROU 3 : Une fonction Python que Claude a le droit d'appeler s'appelle un... (mot anglais)
reponse_3 = "___3___"

# TROU 4 : Le texte entre triple guillemets sous une fonction, lu par Claude pour savoir quand l'utiliser :
reponse_4 = "___4___"

# TROU 5 : Le nom du mode de communication (entrée/sortie standard) entre Claude et le serveur :
reponse_5 = "___5___"

In [ ]:
# ── Vérification du quiz (exécutez sans modifier) ────────────────────────
verifier(1, reponse_1, "8199166184f1")
verifier(2, reponse_2, "948fe603f61d")
verifier(3, reponse_3, "7c9bbe5ec9b3")
verifier(4, reponse_4, "247499209ab0")
verifier(5, reponse_5, "0708da13dbf9");

## Module 2 — Imports, adresses des API et création du serveur

Trois étapes, comme au début de `server.py` :
1. **importer** les outils (une bibliothèque pour parler aux API web, une pour créer un serveur MCP) ;
2. **noter les adresses** des deux API Open-Meteo : le *géocodage* (ville → GPS) et les *prévisions* (GPS → météo) ;
3. **créer l'objet serveur**, en lui donnant le nom qui s'affichera dans Claude Desktop.

### Exercice 2 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | bibliothèque pour faire des appels HTTP (requêtes web) | `requests` ou `pandas` |
| `___2___` | la classe « serveur MCP rapide » (Fast + MCP) | `Flask` ou `FastMCP` |
| `___3___` | « géocodage » en anglais | `geocoding` ou `weather` |
| `___4___` | « prévisions » en anglais | `meteo` ou `forecast` |
| `___5___` | le nom du serveur de la démo (deux mots) | `Meteo Africa` ou `Claude` |

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import ___1___                                   # TROU 1 : bibliothèque pour faire des appels HTTP (requêtes web)
import urllib3                               # sert juste à couper un avertissement SSL gênant sous Windows
from mcp.server.fastmcp import ___2___           # TROU 2 : la classe « serveur MCP rapide » (Fast + MCP)

# Certains PC Windows ne reconnaissent pas le certificat SSL → on coupe l'avertissement pour l'atelier
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Adresses des deux API Open-Meteo (gratuites, sans clé) ──────────────────
URL_GEOCODAGE = "https://___3___-api.open-meteo.com/v1/search"   # TROU 3 : « géocodage » en anglais
URL_METEO     = "https://api.open-meteo.com/v1/___4___"          # TROU 4 : « prévisions » en anglais

# ── Création du serveur ──────────────────────────────────────────────────────
mcp = FastMCP("___5___")                        # TROU 5 : le nom du serveur de la démo (deux mots)

In [ ]:
# ── Vérification (exécutez sans modifier) ────────────────────────────────
print("Trou 1 : OK" if requests.__name__ == "requests" else "Trou 1 : ERREUR")
print("Trou 2 : OK" if FastMCP.__name__ == "FastMCP" else "Trou 2 : ERREUR")
print("Trou 3 : OK" if "geocoding-api" in URL_GEOCODAGE else "Trou 3 : ERREUR : relisez l'adresse de géocodage")
print("Trou 4 : OK" if URL_METEO.endswith("/forecast") else "Trou 4 : ERREUR : relisez l'adresse météo")
print("Trou 5 : OK" if mcp.name == "Meteo Africa" else f"Trou 5 : ERREUR : le serveur s'appelle '{mcp.name}'")

## Module 3 — Traduire les codes météo

L'API ne répond pas « il pleut » : elle répond `61`. C'est un **code WMO** (Organisation météorologique mondiale).
On construit donc un **dictionnaire** : à gauche le code, à droite le texte lisible.

Petit piège réel : les codes ne se suivent pas (0, 1, 2, 3, puis 45…). Aidez-vous des voisins de chaque ligne !

### Exercice 3 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | aucun nuage dans le ciel | `Soleil` ou `Ciel degage` |
| `___2___` | ciel entièrement gris (100 % nuages) | `Couvert` ou `Nuageux` |
| `___3___` | le code de la pluie légère (juste avant 63) | `60` ou `61` |
| `___4___` | éclairs et tonnerre | `Orage` ou `Pluie` |
| `___5___` | le code le plus élevé de l'échelle (deux chiffres identiques) | `100` ou `99` |

In [ ]:
# ── Dictionnaire code WMO → texte lisible ────────────────────────────────────
# (accents volontairement retirés : certaines consoles Windows les affichent mal)
CODES_METEO = {
    0:  "___1___",                   # TROU 1 : aucun nuage dans le ciel
    1:  "Principalement degage",
    2:  "Partiellement nuageux",
    3:  "___2___",                   # TROU 2 : ciel entièrement gris (100 % nuages)
    45: "Brouillard",
    48: "Brouillard givrant",
    51: "Bruine legere",
    53: "Bruine moderee",
    55: "Bruine dense",
    ___3___: "Pluie legere",         # TROU 3 : le code de la pluie légère (juste avant 63)
    63: "Pluie moderee",
    65: "Forte pluie",
    71: "Neige legere",
    73: "Neige moderee",
    75: "Forte neige",
    80: "Averses legeres",
    81: "Averses moderees",
    82: "Averses violentes",
    95: "___4___",                   # TROU 4 : éclairs et tonnerre
    96: "Orage avec grele",
    ___5___: "Orage violent avec grele",   # TROU 5 : le code le plus élevé de l'échelle (deux chiffres identiques)
}

print(len(CODES_METEO), "codes connus")    # → 21 codes connus

In [ ]:
# ── Vérification (exécutez sans modifier) ────────────────────────────────
print("Trou 1", CODES_METEO.get(0),  "→ attendu : ciel sans nuage")
print("Trou 2", CODES_METEO.get(3),  "→ attendu : ciel gris")
print("Trou 3", "OK" if CODES_METEO.get(61) == "Pluie legere" else "ERREUR : code de la pluie légère ?")
print("Trou 4", CODES_METEO.get(95), "→ attendu : tonnerre")
print("Trou 5", "OK" if max(CODES_METEO) == 99 else "ERREUR : code le plus élevé ?")

## Module 4 — Fonction interne n°1 : de la ville aux coordonnées GPS

L'API météo ne comprend pas « Yaoundé », elle veut une **latitude** et une **longitude**.
Cette fonction pose la question à l'API de géocodage.

Le `_` au début du nom signale une **fonction interne** : ce n'est **pas** un outil MCP,
Claude ne la voit pas. Seuls nos outils l'utiliseront.

### Exercice 4 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | combien de résultats on veut ? (seulement le plus pertinent) | `1` ou `10` |
| `___2___` | le verbe HTTP pour LIRE | `post` ou `get` |
| `___3___` | transforme le texte reçu en dictionnaire Python (format de données du web) | `json` ou `text` |
| `___4___` | le mot Python qui veut dire « rien » | `False` ou `None` |
| `___5___` | position du PREMIER élément d'une liste en Python | `0` ou `1` |

In [ ]:
def _geocoder_ville(nom_ville):
    """Transforme un nom de ville en {nom, pays, region, latitude, longitude}, ou None."""

    # ── Ce qu'on envoie à l'API ──────────────────────────────────────────────
    params = {
        "name":     nom_ville,
        "count":    ___1___,          # TROU 1 : combien de résultats on veut ? (seulement le plus pertinent)
        "language": "fr",         # noms de villes et de pays en français
        "format":   "json",
    }

    # ── L'appel web ──────────────────────────────────────────────────────────
    # timeout=10 : on n'attend pas plus de 10 s ; verify=False : contournement SSL Windows
    reponse = requests.___2___(URL_GEOCODAGE, params=params, timeout=10, verify=False)  # TROU 2 : le verbe HTTP pour LIRE
    reponse.raise_for_status()    # si le code HTTP n'est pas 2xx (ex. 500), on lève une erreur tout de suite

    donnees = reponse.___3___()       # TROU 3 : transforme le texte reçu en dictionnaire Python (format de données du web)

    # ── Ville inconnue ? ─────────────────────────────────────────────────────
    # Piège réel : quand la ville n'existe pas, la clé "results" est ABSENTE (pas une liste vide)
    if not donnees.get("results"):
        return ___4___                # TROU 4 : le mot Python qui veut dire « rien »

    r = donnees["results"][___5___]   # TROU 5 : position du PREMIER élément d'une liste en Python

    return {
        "nom":       r["name"],
        "pays":      r["country"],
        "region":    r.get("admin1", ""),   # la région peut manquer → "" par défaut au lieu d'une erreur
        "latitude":  r["latitude"],
        "longitude": r["longitude"],
    }

In [ ]:
# ── Test (exécutez sans modifier) ────────────────────────────────────────
print(_geocoder_ville("Yaounde"))          # → {'nom': 'Yaoundé', 'pays': 'Cameroun', ...}
print(_geocoder_ville("Villequinexistepas"))  # → None

## Module 5 — Fonction interne n°2 : des coordonnées GPS à la météo

On a la latitude et la longitude de Yaoundé. On demande maintenant à l'API météo les valeurs **actuelles**
(température, ressenti, humidité, vent, code météo).

### Exercice 5 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | le paramètre reçu qui contient la latitude | `latitude` ou `lat` |
| `___2___` | le paramètre reçu qui contient la longitude | `lon` ou `longitude` |
| `___3___` | fuseau horaire détecté AUTOmatiquement (heure locale de la ville) | `Europe/Paris` ou `auto` |
| `___4___` | la méthode qui plante si le code HTTP est une erreur (vue au Module 4) | `raise_for_status` ou `status_code` |
| `___5___` | « actuel » en anglais (le même mot que la clé "current" plus haut) | `daily` ou `current` |

In [ ]:
def _appeler_api_meteo(lat, lon):
    """Renvoie le dictionnaire 'current' d'Open-Meteo pour des coordonnées GPS."""

    params = {
        "latitude":      ___1___,     # TROU 1 : le paramètre reçu qui contient la latitude
        "longitude":     ___2___,     # TROU 2 : le paramètre reçu qui contient la longitude
        # Les mesures voulues, séparées par des virgules (sans espace !)
        "current":       "temperature_2m,apparent_temperature,relative_humidity_2m,wind_speed_10m,weather_code",
        "timezone":      "___3___",   # TROU 3 : fuseau horaire détecté AUTOmatiquement (heure locale de la ville)
        "forecast_days": 1,       # seulement aujourd'hui
    }

    reponse = requests.get(URL_METEO, params=params, timeout=10, verify=False)
    reponse.___4___()                 # TROU 4 : la méthode qui plante si le code HTTP est une erreur (vue au Module 4)

    # On ne garde que le sous-dictionnaire des valeurs du moment
    return reponse.json()["___5___"]    # TROU 5 : « actuel » en anglais (le même mot que la clé "current" plus haut)

In [ ]:
# ── Test (exécutez sans modifier) ────────────────────────────────────────
coords = _geocoder_ville("Yaounde")
actuel = _appeler_api_meteo(coords["latitude"], coords["longitude"])
actuel                                    # → {'time': ..., 'temperature_2m': ..., 'weather_code': ...}

## Module 6 — Outil MCP n°1 : `meteo_ville`

C'est ici que la magie MCP opère. Trois ingrédients transforment une fonction ordinaire en **outil pour Claude** :

1. le **décorateur** `@mcp.tool()` posé au-dessus de la fonction : il l'inscrit dans le serveur ;
2. des **annotations de type** (`ville: str`, `-> str`) : Claude sait quoi envoyer et quoi attendre ;
3. une **docstring** claire : c'est le « mode d'emploi » que Claude lit pour décider *quand* appeler l'outil.

> Attention : Un outil MCP renvoie **toujours du texte** : c'est ce texte que Claude lira puis reformulera.

### Exercice 6 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | le décorateur qui transforme la fonction en OUTIL MCP | `tool` ou `function` |
| `___2___` | le type Python d'un texte (abréviation de « string ») | `int` ou `str` |
| `___3___` | ce que renvoie _geocoder_ville quand la ville est inconnue | `None` ou `False` |
| `___4___` | le dictionnaire du Module 3 | `METEO` ou `CODES_METEO` |
| `___5___` | température « apparente » | `apparent_temperature` ou `temperature_2m` |

In [ ]:
@mcp.___1___()                                      # TROU 1 : le décorateur qui transforme la fonction en OUTIL MCP
def meteo_ville(ville: ___2___) -> str:             # TROU 2 : le type Python d'un texte (abréviation de « string »)
    """
    Obtient la météo actuelle et complète d'une ville dans le monde.

    Utilise cet outil quand l'utilisateur :
    - Demande la météo d'une ville spécifique
    - Veut savoir s'il fait chaud, froid, s'il pleut quelque part

    Paramètre :
        ville : Nom de la ville (ex: 'Yaoundé', 'Dakar', 'Paris', 'Abidjan')
    """
    # Étape 1 : ville → GPS
    coords = _geocoder_ville(ville)

    # Étape 2 : cas d'erreur — on renvoie un message utile à Claude plutôt que de planter
    if coords is ___3___:                           # TROU 3 : ce que renvoie _geocoder_ville quand la ville est inconnue
        return f"Ville '{ville}' introuvable. Essayez sans accents (ex: 'Yaounde')."

    # Étape 3 : GPS → météo
    actuel = _appeler_api_meteo(coords["latitude"], coords["longitude"])

    # Étape 4 : code → texte. .get(clé, défaut) évite une erreur si le code est inconnu du dictionnaire
    description = ___4___.get(actuel["weather_code"], "Code inconnu")   # TROU 4 : le dictionnaire du Module 3

    # Étape 5 : on construit le texte que Claude va lire
    return (
        f"Meteo a {coords['nom']}, {coords['pays']}\n"
        f"{'=' * 42}\n"
        f"Temperature     : {actuel['temperature_2m']} C (ressentie {actuel['___5___']} C)\n"   # TROU 5 : température « apparente »
        f"Humidite        : {actuel['relative_humidity_2m']} %\n"
        f"Vent            : {actuel['wind_speed_10m']} km/h\n"
        f"Conditions      : {description}\n"
        f"Heure locale    : {actuel['time']}"
    )

In [ ]:
# ── Test : c'est EXACTEMENT ce que Claude recevra (exécutez sans modifier) ─
print(meteo_ville("Yaounde"))
print()
print(meteo_ville("Atlantide"))           # → message d'erreur propre, pas un plantage

## Module 7 — Outil MCP n°2 : `comparer_meteo`

Scénario : *« Claude, j'ai des collègues à Dakar, Abidjan et Paris, où fait-il le plus chaud ? »*
L'outil reçoit une **liste** de villes, les parcourt une par une et retient la plus chaude et la plus froide.

### Exercice 7 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | le maximum annoncé dans la docstring | `10` ou `5` |
| `___2___` | la liste reçue en paramètre | `villes` ou `liste` |
| `___3___` | mot-clé : « passe directement à la ville suivante » | `break` ou `continue` |
| `___4___` | le signe « plus grand que » | `>` ou `<` |
| `___5___` | la méthode qui COLLE les lignes avec un retour à la ligne | `split` ou `join` |

In [ ]:
@mcp.tool()
def comparer_meteo(villes: list) -> str:
    """
    Compare la météo actuelle entre plusieurs villes (maximum 5).
    Utilise cet outil quand l'utilisateur veut savoir quelle ville est la plus chaude
    ou la plus froide, ou compare des destinations de voyage.

    Paramètre :
        villes : Liste de noms de villes (ex: ['Dakar', 'Abidjan', 'Paris'])
    """
    # Garde-fou : chaque ville = 2 appels web, on limite pour rester rapide
    if len(villes) > ___1___:                       # TROU 1 : le maximum annoncé dans la docstring
        return "Maximum 5 villes a la fois."

    lignes = [f"COMPARAISON METEO — {len(villes)} villes", "=" * 52]
    temp_max, temp_min = None, None             # None = « pas encore de valeur » au départ
    ville_chaude, ville_froide = "", ""

    for nom_ville in ___2___:                       # TROU 2 : la liste reçue en paramètre
        coords = _geocoder_ville(nom_ville)
        if coords is None:
            lignes.append(f"  {nom_ville} : INTROUVABLE")
            ___3___                                 # TROU 3 : mot-clé : « passe directement à la ville suivante »

        actuel = _appeler_api_meteo(coords["latitude"], coords["longitude"])
        temp = actuel["temperature_2m"]
        desc = CODES_METEO.get(actuel["weather_code"], "Code inconnu")
        lignes.append(f"  {coords['nom']:<15} {temp:>5} C  |  {desc}")

        # Mise à jour des records
        if temp_max is None or temp ___4___ temp_max:   # TROU 4 : le signe « plus grand que »
            temp_max, ville_chaude = temp, coords["nom"]
        if temp_min is None or temp < temp_min:
            temp_min, ville_froide = temp, coords["nom"]

    if temp_max is not None:
        lignes.append("-" * 52)
        lignes.append(f"Plus chaude : {ville_chaude} ({temp_max} C)")
        lignes.append(f"Plus froide : {ville_froide} ({temp_min} C)")

    return "\n".___5___(lignes)                    # TROU 5 : la méthode qui COLLE les lignes avec un retour à la ligne

In [ ]:
# ── Test (exécutez sans modifier) ────────────────────────────────────────
print(comparer_meteo(["Dakar", "Abidjan", "Paris", "Atlantide"]))

## Module 8 — Outil MCP n°3 : `conseil_tenue`

Scénario : *« Claude, qu'est-ce que je mets pour aller à Douala aujourd'hui ? »*
L'outil choisit des conseils selon des **paliers de température** (du plus chaud au plus froid),
puis ajoute un conseil si le **code météo** annonce de la pluie.

### Exercice 8 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | palier « très chaud » (5 de plus que 30) | `35` ou `30` |
| `___2___` | mot-clé Python : « sinon, si... » | `if` ou `elif` |
| `___3___` | mot-clé Python : « dans tous les autres cas » | `else` ou `then` |
| `___4___` | la liste qu'on vient de définir juste au-dessus | `CODES_METEO` ou `codes_pluie` |
| `___5___` | AJOUTER un élément à la fin d'une liste | `append` ou `remove` |

In [ ]:
@mcp.tool()
def conseil_tenue(ville: str) -> str:
    """
    Donne des conseils vestimentaires adaptés à la météo actuelle d'une ville.
    Utilise cet outil quand l'utilisateur demande quoi porter, s'il faut un manteau,
    un parapluie ou de la crème solaire, ou prépare ses bagages.
    """
    coords = _geocoder_ville(ville)
    if coords is None:
        return f"Ville '{ville}' introuvable."

    actuel = _appeler_api_meteo(coords["latitude"], coords["longitude"])
    temp = actuel["temperature_2m"]
    code = actuel["weather_code"]

    # ── Paliers de température : on teste du plus chaud au plus froid ──────
    if temp >= ___1___:                             # TROU 1 : palier « très chaud » (5 de plus que 30)
        conseils = ["Vetements tres legers (coton, lin)", "Creme solaire indice 30+", "Beaucoup d'eau"]
    ___2___ temp >= 20:                             # TROU 2 : mot-clé Python : « sinon, si... »
        conseils = ["Tenue legere : t-shirt", "Lunettes de soleil"]
    elif temp >= 10:
        conseils = ["Pull ou veste indispensable"]
    ___3___:                                        # TROU 3 : mot-clé Python : « dans tous les autres cas »
        conseils = ["Manteau chaud", "Echarpe et gants"]

    # ── Conseil pluie : liste des codes qui veulent dire « il pleut » ───────
    codes_pluie = [51, 53, 55, 61, 63, 65, 80, 81, 82]
    if code in ___4___:                             # TROU 4 : la liste qu'on vient de définir juste au-dessus
        conseils.___5___("Parapluie OBLIGATOIRE — il pleut !")   # TROU 5 : AJOUTER un élément à la fin d'une liste

    lignes = [f"CONSEIL TENUE — {coords['nom']}, {coords['pays']}",
              f"Conditions : {temp} C | {CODES_METEO.get(code, 'Code inconnu')}", ""]
    lignes += [f"  - {c}" for c in conseils]    # un tiret devant chaque conseil
    return "\n".join(lignes)

In [ ]:
# ── Test (exécutez sans modifier) ────────────────────────────────────────
print(conseil_tenue("Douala"))

## Module 9 — Outil MCP n°4 : `meteo_resume`

Le dernier outil renvoie **une seule ligne** : pratique quand Claude doit être bref.
Cette fois, c'est à vous de reposer vous-même les briques vues dans les modules précédents.

### Exercice 9 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | même décorateur qu'aux Modules 6, 7 et 8 | `function` ou `tool` |
| `___2___` | un outil MCP renvoie toujours du... | `str` ou `list` |
| `___3___` | la fonction interne « ville → GPS » (Module 4) | `meteo_ville` ou `_geocoder_ville` |
| `___4___` | la fonction interne « GPS → météo » (Module 5) | `_appeler_api_meteo` ou `_geocoder_ville` |
| `___5___` | la clé de l'humidité relative (voir le test du Module 5) | `humidity` ou `relative_humidity_2m` |

In [ ]:
@mcp.___1___()                                        # TROU 1 : même décorateur qu'aux Modules 6, 7 et 8
def meteo_resume(ville: str) -> ___2___:              # TROU 2 : un outil MCP renvoie toujours du...
    """
    Donne un résumé météo très court, en une seule ligne, pour une ville.
    Utilise cet outil quand l'utilisateur veut une réponse rapide et concise.
    """
    coords = ___3___(ville)                           # TROU 3 : la fonction interne « ville → GPS » (Module 4)
    if coords is None:
        return f"'{ville}' : ville introuvable"

    actuel = ___4___(coords["latitude"], coords["longitude"])   # TROU 4 : la fonction interne « GPS → météo » (Module 5)

    temp = actuel["temperature_2m"]
    hum  = actuel["___5___"]                          # TROU 5 : la clé de l'humidité relative (voir le test du Module 5)
    desc = CODES_METEO.get(actuel["weather_code"], "Code inconnu")

    return f"{coords['nom']} ({coords['pays']}) : {temp} C, {desc}, Humidite {hum}%"

In [ ]:
# ── Test (exécutez sans modifier) ────────────────────────────────────────
for v in ["Yaounde", "Dakar", "Nairobi"]:
    print(meteo_resume(v))

## Module 10 — Assembler le fichier `mon_serveur.py`

Jusqu'ici, tout vit **dans ce notebook**. Or Claude Desktop lance un **fichier Python** indépendant.
Cette cellule récupère le code de VOS fonctions (celles que vous venez de compléter)
et les écrit dans `mon_serveur.py`, avec, tout en bas, la ligne qui démarre le serveur.

> Astuce : Si un outil ne marche pas plus tard, c'est dans le module correspondant qu'il faut corriger, puis relancer cette cellule.

### Exercice 10 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | le nom du fichier à créer (annoncé dans le titre) | `mon_serveur.py` ou `server.py` |
| `___2___` | le 4e outil, celui du Module 9 | `meteo_ville` ou `meteo_resume` |
| `___3___` | « obtenir la source » en anglais (get + source) | `getsource` ou `getfile` |
| `___4___` | le transport utilisé par Claude Desktop (quiz du Module 1) | `http` ou `stdio` |
| `___5___` | la lettre du mode ÉCRITURE | `w` ou `r` |

In [ ]:
import inspect                                   # permet de lire le code source d'une fonction
import pprint                                    # affiche joliment un dictionnaire (sur plusieurs lignes)

FICHIER = "___1___"                                  # TROU 1 : le nom du fichier à créer (annoncé dans le titre)

# ── En-tête : imports, adresses, dictionnaire, création du serveur ──────────
entete = f'''"""Serveur MCP Meteo Africa — généré depuis l'exercice MCP de A à Z."""
import requests
import urllib3
from mcp.server.fastmcp import FastMCP

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

URL_GEOCODAGE = {URL_GEOCODAGE!r}
URL_METEO     = {URL_METEO!r}
CODES_METEO   = {pprint.pformat(CODES_METEO)}

mcp = FastMCP({mcp.name!r})
'''

# ── Corps : les 2 fonctions internes + les 4 outils, dans cet ordre ─────────
fonctions = [_geocoder_ville, _appeler_api_meteo,
             meteo_ville, comparer_meteo, conseil_tenue, ___2___]   # TROU 2 : le 4e outil, celui du Module 9
corps = "\n\n".join(inspect.___3___(f) for f in fonctions)        # TROU 3 : « obtenir la source » en anglais (get + source)

# ── Pied : démarrage du serveur quand on lance le fichier ───────────────────
pied = '''
if __name__ == "__main__":
    mcp.run(transport="___4___")   # TROU 4 : le transport utilisé par Claude Desktop (quiz du Module 1)
'''

# "w" = write : on crée le fichier (ou on l'écrase s'il existe déjà)
with open(FICHIER, "___5___", encoding="utf-8") as f:   # TROU 5 : la lettre du mode ÉCRITURE
    f.write(entete + "\n\n" + corps + "\n" + pied)

print(f"OK {FICHIER} écrit ({len(fonctions)} fonctions)")

In [ ]:
# ── Vérification : le fichier est-il du Python valide ? (exécutez sans modifier)
import ast
source = open("mon_serveur.py", encoding="utf-8").read()
ast.parse(source)                                # plante si le fichier contient une erreur de syntaxe
print(f"OK Syntaxe valide — {source.count('@mcp.tool()')} outils déclarés (attendu : 4)")
print("OK Transport stdio" if 'transport="stdio"' in source else "ERREUR : Transport : à vérifier (Module 10, trou Trou 4)")

## Module 11 — Tester le serveur comme Claude le fait

Avant de brancher Claude, on joue nous-mêmes le rôle du **client MCP**. On refait exactement les 4 gestes de Claude Desktop :

1. **lancer** `mon_serveur.py` en arrière-plan et s'y connecter par **stdio** ;
2. **initialiser** la session (poignée de main : « bonjour, je parle MCP ») ;
3. **lister les outils** disponibles (Claude découvre les 4 outils et leur docstring) ;
4. **appeler un outil** avec des arguments.

> Le mot `await` veut dire « attends la réponse avant de continuer » : le serveur est un autre programme.

### Exercice 11 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | le fichier créé au Module 10 | `server.py` ou `mon_serveur.py` |
| `___2___` | la fonction importée juste au-dessus (client stdio) | `stdio_client` ou `http_client` |
| `___3___` | « initialiser » en anglais | `start` ou `initialize` |
| `___4___` | « lister les outils » en anglais (avec un _) | `list_tools` ou `get_tools` |
| `___5___` | l'outil de la météo complète | `meteo_resume` ou `meteo_ville` |

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# ── Comment lancer le serveur : quel Python, quel fichier ────────────────────
parametres = StdioServerParameters(
    command=sys.executable,                    # le même Python que ce notebook
    args=["___1___"],                              # TROU 1 : le fichier créé au Module 10
)

async def tester_serveur():
    # ── Geste 1 : lancer le serveur et ouvrir le canal stdio ────────────────
    # Piège Jupyter : le serveur écrit ses logs sur « stderr », mais Jupyter n'a pas de vrai stderr
    # (erreur « UnsupportedOperation: fileno ») → on envoie ces logs dans un fichier à part.
    journal = open("mon_serveur.log", "w", encoding="utf-8")
    async with ___2___(parametres, errlog=journal) as (lecture, ecriture):   # TROU 2 : la fonction importée juste au-dessus (client stdio)
        async with ClientSession(lecture, ecriture) as session:

            # ── Geste 2 : la poignée de main ────────────────────────────────
            await session.___3___()                                  # TROU 3 : « initialiser » en anglais

            # ── Geste 3 : découvrir les outils ──────────────────────────────
            outils = await session.___4___()                         # TROU 4 : « lister les outils » en anglais (avec un _)
            for outil in outils.tools:
                print("-", outil.name, "—", outil.description.strip().splitlines()[0])

            # ── Geste 4 : appeler l'outil qui répond à notre question fil rouge ─
            resultat = await session.call_tool("___5___", {"ville": "Yaounde"})   # TROU 5 : l'outil de la météo complète
            print()
            print(resultat.content[0].text)

await tester_serveur()                         # dans Jupyter, on peut « await » directement

> **Si la cellule plante** alors que les tests des Modules 6 à 9 fonctionnaient : ouvrez `mon_serveur.log`
> (les erreurs du serveur y sont écrites). Si le problème vient de Jupyter lui-même, passez au Module 12 :
> Claude Desktop lance le serveur sans passer par Jupyter.

## Module 12 — Brancher le serveur sur Claude Desktop

Claude Desktop lit un fichier de configuration `claude_desktop_config.json` qui dit :
*« au démarrage, lance tel programme avec tel fichier »*.
La cellule ci-dessous **fabrique** le texte à coller — elle ne modifie rien sur votre ordinateur.

### Exercice 12 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | la clé racine : « serveurs MCP » (mcp + Servers, collés) | `mcpServers` ou `servers` |
| `___2___` | le chemin du Python de ce notebook (vu au Module 0 : sys.…) | `"python"` ou `sys.executable` |
| `___3___` | la variable qui contient le chemin du serveur (2 lignes plus haut) | `chemin_serveur` ou `FICHIER` |
| `___4___` | la fonction qui transforme le dictionnaire en texte (dump + s) | `loads` ou `dumps` |
| `___5___` | nombre d'espaces pour décaler chaque niveau (classique : deux) | `2` ou `4` |

In [ ]:
import json
import os

chemin_serveur = os.path.abspath("mon_serveur.py")   # chemin COMPLET : Claude ne sait pas où est ce notebook

config = {
    "___1___": {                                          # TROU 1 : la clé racine : « serveurs MCP » (mcp + Servers, collés)
        "meteo-africa": {                             # nom libre, affiché dans Claude Desktop
            "command": ___2___,                           # TROU 2 : le chemin du Python de ce notebook (vu au Module 0 : sys.…)
            "args": [___3___],                            # TROU 3 : la variable qui contient le chemin du serveur (2 lignes plus haut)
        }
    }
}

# json.dumps = transformer le dictionnaire en TEXTE JSON ; indent = nombre d'espaces d'indentation
texte = json.___4___(                   # TROU 4 : la fonction qui transforme le dictionnaire en texte (dump + s)
    config,
    indent=___5___,                     # TROU 5 : nombre d'espaces pour décaler chaque niveau (classique : deux)
    ensure_ascii=False,             # garde les accents lisibles dans les chemins
)
print(texte)

### Coller la configuration dans Claude Desktop

1. Ouvrez Claude Desktop → **Paramètres → Développeur → Modifier la configuration**.
2. Le fichier `claude_desktop_config.json` s'ouvre (sous Windows : `%APPDATA%\Claude\claude_desktop_config.json`).
3. Collez le texte affiché ci-dessus. Attention : S'il contient déjà un bloc `"mcpServers"`, ajoutez seulement
   l'entrée `"meteo-africa"` à l'intérieur, sans supprimer les autres serveurs.
4. **Quittez complètement** Claude Desktop (icône de la barre des tâches → Quitter) puis relancez-le.
5. Dans une nouvelle conversation, l'icône des outils (le marteau) doit indiquer **4 outils**.

### Phrases à tester

```
Quelle est la météo à Yaoundé en ce moment ?
Compare la météo entre Dakar, Abidjan et Paris.
Qu'est-ce que je dois mettre pour aller à Douala aujourd'hui ?
Donne-moi un résumé en une ligne de la météo à Nairobi.
```

Observez **quel outil** Claude choisit pour chaque phrase : il se fie uniquement aux docstrings que vous avez lues.

## Exercice Final — Le récap en phrases

Dernier exercice, sans code : complétez le résumé de tout ce que vous venez de construire.

### Exercice 13 — 5 trous à remplacer

| Trou | Ce qu'il faut écrire à la place | Choisir entre |
|---|---|---|
| `___1___` | La ligne « @mcp.tool() » posée au-dessus d'une fonction s'appelle un... | `commentaire` ou `décorateur` |
| `___2___` | Pour choisir le bon outil, Claude lit la... de chaque fonction. | `docstring` ou `titre` |
| `___3___` | Les fonctions qui commencent par _ (ex. _geocoder_ville) ne sont PAS des outils : elles sont... | `publiques` ou `internes` |
| `___4___` | Pour transformer « Yaoundé » en latitude/longitude, on appelle l'API de... | `géocodage` ou `prévision` |
| `___5___` | Le fichier qui dit à Claude Desktop quel serveur lancer est au format... | `csv` ou `json` |

In [ ]:
# ── Complétez chaque phrase (un mot entre les guillemets) ───────────────────

# TROU 1 : La ligne « @mcp.tool() » posée au-dessus d'une fonction s'appelle un...
recap_1 = "___1___"

# TROU 2 : Pour choisir le bon outil, Claude lit la... de chaque fonction.
recap_2 = "___2___"

# TROU 3 : Les fonctions qui commencent par _ (ex. _geocoder_ville) ne sont PAS des outils : elles sont...
recap_3 = "___3___"

# TROU 4 : Pour transformer « Yaoundé » en latitude/longitude, on appelle l'API de...
recap_4 = "___4___"

# TROU 5 : Le fichier qui dit à Claude Desktop quel serveur lancer est au format...
recap_5 = "___5___"

In [ ]:
# ── Vérification finale (exécutez sans modifier) ────────────────────────
score = sum([
    verifier(1, recap_1, "353ae4184555"),
    verifier(2, recap_2, "247499209ab0"),
    verifier(3, recap_3, "fff44aba1b8e"),
    verifier(4, recap_4, "7c9a30b8596c"),
    verifier(5, recap_5, "02bd175f3297"),
])
print(f"\nScore : {score}/5", "Bravo, vous avez construit un serveur MCP de A à Z !" if score == 5 else "")

## OK Ce qu'on a appris

- Un **serveur MCP** = un programme Python qui expose des **outils** (`@mcp.tool()`).
- Claude choisit l'outil grâce à la **docstring** et aux **types** (`ville: str -> str`).
- Un outil renvoie toujours du **texte**, que Claude reformule pour l'utilisateur.
- Les fonctions **internes** (`_geocoder_ville`, `_appeler_api_meteo`) font le travail technique, invisible pour Claude.
- Claude Desktop lance le serveur en **stdio**, grâce à `claude_desktop_config.json`.

**Pour aller plus loin :** comparez votre `mon_serveur.py` avec `../server.py` (la version complète de la démo).
Idée d'extension : ajoutez un 5e outil `prevision_demain(ville)` en demandant `daily` au lieu de `current` à l'API.